# CS453 - Two-Week Individual Project Workbook

**Name:** أحمد مفتاح العجيلي
**Student ID:** 09231026  
**Project:** Cleaning Robot

Read the two-week student guide first. This workbook supplies only map data and a variant helper, not search or agent solutions. Complete your own explanations and implementations. Save this notebook in your private GitHub repository as project.ipynb and follow docs/github-colab-guide.md. Submit the repository and final commit links in Classroom. The full course project continues after the professor returns.


## 1. Task, PEAS and environment

Explain your chosen task, inputs, output and success criteria. Complete PEAS and justify observability, determinism, sequential/episodic, static/dynamic, discrete/continuous and number of agents for this initial simulation. What are three scope limits?


المهمة، المدخلات، المخرجات، ومعايير النجاح:

المهمة : فكرة مشروعي هي روبوت تنظيف، يعني العميل (الروبوت) لازم يتحرك من نقطة البداية لين يوصل لغرفة معينة مطلوبة منه، وينظفها.

المدخلات : مكان البداية اللي هو ('S')، الغرفة الهدف اللي لازم تتنظف وهي ('D')، خريطة الأماكن اللي يتحرك فيها، وحالة الغرفة (نظيفة أو مسخة).

المخرجات : سلسلة من الحركات الصحيحة (تنقلات بين الغرف) وتختم بحركة التنظيف باش نحقق الهدف.

معايير النجاح : إن الروبوت يوصل للغرفة ('D') وينظفها بأقل عدد ممكن من الخطوات، ومن غير ما يدير أي حركة غالطة أو ممنوعة.

وصف بيئة العمل (PEAS):

مقياس الأداء (Performance): تنظيف الغرفة المطلوبة بأقل تكلفة مسار ممكنة، وتجنب أي حركات غير قانونية.

البيئة (Environment): خريطة ثابتة فيها 8 أماكن (من S لـ G)، والطريق بين E و G ملغية حسب الـ Variant بتاعي.

المشغلات (Actuators): حركة التنقل من مكان للثاني، وحركة التنظيف.

المستشعرات (Sensors): مستشعر لمعرفة المكان الحالي، ومستشعر لمعرفة حالة الغرفة (نظيفة أو لا).

خصائص البيئة:

يمكن ملاحظتها بالكامل (Fully Observable): الروبوت ديما عارف مكانه وين بالضبط، وعارف حالة الغرفة المطلوبة.

حتمية (Deterministic): لما الروبوت يقرر يتحرك لغرفة جنبها، حيوصلها أكيد.

تتابعية (Sequential): أي خطوة ياخذها الروبوت توا، حتأثر على الخطوات الجاية وتكلفة المسار بالكامل.

ثابتة (Static): الخريطة ما تتغيرش أثناء حركة الروبوت، والغرفة تقعد مسخة لين ينظفها هو.

منفصلة (Discrete): عدد الأماكن محدود ومعروف، والحركات اللي يقدر يديرها الروبوت محدودة وواضحة.

عميل واحد (Single Agent): الروبوت هو الوحيد اللي يشتغل في الخريطة هذي.

ثلاث حدود للمشروع (Scope Limits):

مافيش أي تعقيدات فيزيائية؛ يعني مافيش اعتبار لشحن البطارية أو المسافات الحقيقية (كل حركة تكلفتها 1 وخلاص).

الروبوت ينفذ مهمة وحدة بس (تنظيف غرفة وحدة)، مش سلسلة من طلبات التنظيف ورا بعض.

البيئة مبسطة على شكل عُقد (Nodes) وطرق، فما فيش حاجة لمعالجة صور أو حساسات حقيقية.


## 2. Representation

Define initial state, legal actions, result/transition, goal test and costs. Include task status as well as location. Define service legality and repeated-service behavior. State whether the requested task is initially complete.


In [2]:
# Supplied data/helper: cite CS453 two-week project workbook if adapted.
from copy import deepcopy
import platform

REFERENCE_GRAPH = {
    "S": ["A", "B"],
    "A": ["S", "C", "D"],
    "B": ["S", "E"],
    "C": ["A", "F"],
    "D": ["A", "G"],
    "E": ["B", "G"],
    "F": ["C", "G"],
    "G": ["D", "E", "F"],
}
VARIANTS = [("A", "D", "G"), ("B", "E", "F"),
            ("C", "F", "E"), ("E", "G", "D")]

def make_variant(student_id):
    if not str(student_id).strip():
        raise ValueError("Enter your student ID first")
    variant = sum(ord(ch) for ch in str(student_id).strip()) % 4
    graph = deepcopy(REFERENCE_GRAPH)
    u, v, goal = VARIANTS[variant]
    graph[u].remove(v)
    graph[v].remove(u)
    return {"variant": variant, "graph": graph, "start": "S",
            "goal": goal, "removed_edge": (u, v)}

STUDENT_ID = "09231026"  # TODO: your student ID as text, preserving leading zeros.
if STUDENT_ID.strip():
    assigned = make_variant(STUDENT_ID)
    print(assigned)
else:
    print("Unfinished template: enter your ID to generate your variant.")
print("Python:", platform.python_version())


{'variant': 3, 'graph': {'S': ['A', 'B'], 'A': ['S', 'C', 'D'], 'B': ['S', 'E'], 'C': ['A', 'F'], 'D': ['A', 'G'], 'E': ['B'], 'F': ['C', 'G'], 'G': ['D', 'F']}, 'start': 'S', 'goal': 'D', 'removed_edge': ('E', 'G')}
Python: 3.13.18


### Supplied heuristic data for Task 6

These are shortest distances in the reference graph with all original roads open. Your assigned variant removes a road, so those relaxed-world distances cannot overestimate the remaining route cost. They are supplied task data; students implement the actual BFS and A* algorithms. For goal F: S=3, A=2, B=3, C=1, D=2, E=2, F=0, G=1. No extra hand-trace task is required.

In [3]:
# Supplied heuristic data: h is distance in the original graph before a road closure.
# Removing roads cannot make a route shorter, so these values remain lower bounds.
# h(goal) = 0. These tables apply to the original graph and its edge-deletion variants.
# If you add shortcuts/change costs, re-justify h or use h = 0 instead.
HEURISTICS = {'A': {'A': 0, 'B': 2, 'C': 1, 'D': 1, 'E': 3, 'F': 2, 'G': 2, 'S': 1}, 'B': {'A': 2, 'B': 0, 'C': 3, 'D': 3, 'E': 1, 'F': 3, 'G': 2, 'S': 1}, 'C': {'A': 1, 'B': 3, 'C': 0, 'D': 2, 'E': 3, 'F': 1, 'G': 2, 'S': 2}, 'D': {'A': 1, 'B': 3, 'C': 2, 'D': 0, 'E': 2, 'F': 2, 'G': 1, 'S': 2}, 'E': {'A': 3, 'B': 1, 'C': 3, 'D': 2, 'E': 0, 'F': 2, 'G': 1, 'S': 2}, 'F': {'A': 2, 'B': 3, 'C': 1, 'D': 2, 'E': 2, 'F': 0, 'G': 1, 'S': 3}, 'G': {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1, 'F': 1, 'G': 0, 'S': 3}, 'S': {'A': 1, 'B': 1, 'C': 2, 'D': 2, 'E': 2, 'F': 3, 'G': 3, 'S': 0}}
print("Heuristic tables available for targets:", ", ".join(sorted(HEURISTICS)))


Heuristic tables available for targets: A, B, C, D, E, F, G, S


## 4. Implement and test the simulator

Write functions for legal moves, state updates, service action and task completion. Reject invalid actions clearly. Do not mutate shared map data between tests. Search plans the route; the simulator verifies/exercises its moves and final service.


In [4]:
# TODO: implement your chosen agent simulator with explicit inputs/outputs.
# Do not implement a physical robot or UI.
# 1. Environment & Graph Definition
# بناء الخريطة وتحديد الطريق الملغي حسب المسار (v=3)
# إزالة الطريق E-G
graph = {
    'S': ['A', 'B'],
    'A': ['S', 'C', 'D'],
    'B': ['S', 'E'],
    'C': ['A', 'F'],
    'D': ['A', 'G'],
    'E': ['B'],  # تمت إزالة G
    'F': ['C', 'G'],
    'G': ['D', 'F'] # تمت إزالة E
}

# 2. Simulator implementation
def initial_state():
    # الروبوت يبدا في S، والغرفة الهدف (D) متسخة (False)
    return ('S', False)

def goal_test(state, target_room):
    location, is_clean = state
    # الهدف يتحقق لو الروبوت في الغرفة المطلوبة ونظفها
    return location == target_room and is_clean == True

def get_legal_actions(state, target_room):
    location, is_clean = state
    actions = []

    # 1. حركات التنقل (Movement): مسموحة ديما للغرف المتصلة
    for neighbor in graph[location]:
        actions.append(('Move', neighbor))

    # 2. حركة التنظيف (Cleaning):
    # مسموحة فقط في الغرفة الهدف، وفقط لو كانت مسخة
    if location == target_room and not is_clean:
        actions.append(('Clean', location))

    return actions

def get_result(state, action):
    location, is_clean = state
    action_type, action_target = action

    # تحديث الحالة بناءً على نوع الحركة
    if action_type == 'Move':
        # يتغير المكان، وحالة التنظيف تقعد زي ما هي
        return (action_target, is_clean)

    elif action_type == 'Clean':
        # لو كانت الحركة تنظيف (وهي قانونية حسب الدالة اللي قبلها)، تتغير حالة الغرفة لـ True
        return (location, True)

# --- Test the simulator ---
print("Testing Simulator Logic:")
test_target = 'D'
current = initial_state()
print(f"Initial State: {current}")

# اختبار حركة التنقل
actions = get_legal_actions(current, test_target)
print(f"Legal actions from {current}: {actions}")
if ('Move', 'A') in actions:
    current = get_result(current, ('Move', 'A'))
    print(f"After moving to A: {current}")

# اختبار الوصول للهدف والتنظيف
current = ('D', False)
print(f"\nState at target room before cleaning: {current}")
actions = get_legal_actions(current, test_target)
print(f"Legal actions: {actions}")
if ('Clean', 'D') in actions:
    current = get_result(current, ('Clean', 'D'))
    print(f"After cleaning: {current}")
    print(f"Goal Reached? {goal_test(current, test_target)}")

Testing Simulator Logic:
Initial State: ('S', False)
Legal actions from ('S', False): [('Move', 'A'), ('Move', 'B')]
After moving to A: ('A', False)

State at target room before cleaning: ('D', False)
Legal actions: [('Move', 'A'), ('Move', 'G'), ('Clean', 'D')]
After cleaning: ('D', True)
Goal Reached? True


## 5. Uninformed search: BFS

Implement BFS using a FIFO queue, repeated-state tracking and route reconstruction. Explain why it is uninformed and why it finds a minimum-move path on these unit-cost graphs. Show the route returned for your assigned mission and the agent executing its service action. Handle start=goal, no route and invalid locations correctly. No separate test-case table or results-analysis task is required.

In [5]:
# TODO: implement and explain BFS, then show your assigned mission.

from collections import deque

def bfs(start_state, target_room):
    # التعامل مع الأماكن غير الصالحة
    if target_room not in graph or start_state[0] not in graph:
        return None

    # التعامل لو كانت البداية هي نفسها الهدف
    if goal_test(start_state, target_room):
        return []

    # الطابور (FIFO Queue) يخزن: (الحالة، مسار الحركات)
    frontier = deque([(start_state, [])])
    explored = set() # تتبع الحالات المكررة

    while frontier:
        current_state, path = frontier.popleft()

        # لو وصلنا للهدف
        if goal_test(current_state, target_room):
            return path

        if current_state not in explored:
            explored.add(current_state)

            # بناء المسار للحركات المسموحة
            for action in get_legal_actions(current_state, target_room):
                next_state = get_result(current_state, action)
                if next_state not in explored:
                    frontier.append((next_state, path + [action]))

    return None # في حال عدم وجود مسار

# --- التفسير النظري وطباعة النتيجة ---
print("تفسير BFS:")
print("خوارزمية BFS تعتبر 'غير موجهة' (uninformed) لأنها ما تستخدمش أي معلومات مساعدة (heuristics) أو أي فكرة عن مسافة الهدف. هي تبحث مستوى بمستوى.")
print("وهي تضمن إيجاد مسار بأقل عدد من الحركات في الخرائط اللي تكلفتها متساوية (unit-cost)، لأنها تستكشف كل المسارات اللي طولها k قبل ما تفكر في المسارات اللي طولها k+1.\n")

print("--- تنفيذ مهمة BFS ---")
start = initial_state()
target = 'D'
bfs_route = bfs(start, target)

if bfs_route is not None:
    print(f"تم إيجاد مسار في {len(bfs_route)} خطوات:")
    for step in bfs_route:
        print(f" -> {step}")
else:
    print("لم يتم إيجاد مسار.")

تفسير BFS:
خوارزمية BFS تعتبر 'غير موجهة' (uninformed) لأنها ما تستخدمش أي معلومات مساعدة (heuristics) أو أي فكرة عن مسافة الهدف. هي تبحث مستوى بمستوى.
وهي تضمن إيجاد مسار بأقل عدد من الحركات في الخرائط اللي تكلفتها متساوية (unit-cost)، لأنها تستكشف كل المسارات اللي طولها k قبل ما تفكر في المسارات اللي طولها k+1.

--- تنفيذ مهمة BFS ---
تم إيجاد مسار في 3 خطوات:
 -> ('Move', 'A')
 -> ('Move', 'D')
 -> ('Clean', 'D')


## 6. Informed search: A*

Implement A* with f(n) = g(n) + h(n), using a priority queue, best-cost tracking and parent reconstruction. Use the supplied HEURISTICS table for your assigned goal. Explain what g and h mean, why the supplied h is admissible on the provided variant, and when it would need revising. Test the goal on removal from the frontier. Show the assigned route and execute the service action. A* and BFS should find the same minimum move count here; their paths may differ. No separate comparison table or limitations essay is required.

In [6]:
# TODO: implement and explain A*, then show your assigned mission.

import heapq

# جدول Heuristics للهدف D المعطى في المشروع
heuristic_D = {'A': 1, 'B': 3, 'C': 2, 'D': 0, 'E': 2, 'F': 2, 'G': 1, 'S': 2}

def a_star(start_state, target_room):
    count = 0 # لحل مشكلة التساوي في قيمة f
    # طابور الأولويات يخزن: (f_cost, count, state, path, g_cost)
    frontier = []
    heapq.heappush(frontier, (heuristic_D[start_state[0]], count, start_state, [], 0))

    # تتبع أفضل تكلفة للوصول لكل حالة
    best_g = {start_state: 0}

    while frontier:
        f, _, current_state, path, g = heapq.heappop(frontier)

        # اختبار الهدف عند الإزالة من الطابور (removal from frontier)
        if goal_test(current_state, target_room):
            return path

        # لو لقينا مسار أفضل مسبقاً، نتجاهل هذا المسار
        if g > best_g.get(current_state, float('inf')):
            continue

        for action in get_legal_actions(current_state, target_room):
            next_state = get_result(current_state, action)
            new_g = g + 1 # تكلفة الوحدة = 1

            if new_g < best_g.get(next_state, float('inf')):
                best_g[next_state] = new_g

                # حساب h: لو المهمة تمت (True) تصبح 0، وإلا نأخذها من الجدول
                h = 0 if next_state[1] else heuristic_D[next_state[0]]
                new_f = new_g + h

                count += 1
                heapq.heappush(frontier, (new_f, count, next_state, path + [action], new_g))

    return None

# --- التفسير النظري وطباعة النتيجة ---
print("تفسير A*:")
print("الـ g(n) يمثل التكلفة الفعلية من نقطة البداية للحالة الحالية n.")
print("الـ h(n) يمثل التكلفة التقديرية (الـ heuristic) من الحالة n للهدف.")
print("قيمة h المعطاة تعتبر مقبولة (admissible) لأنها تمثل أقصر مسار في الخريطة الأصلية. وبما أن مسارنا انحذف منه طريق (E-G)، المسافات الفعلية يا إما تقعد زي ما هي أو تزيد، وبالتالي قيمة h مستحيل تبالغ في التقدير (never overestimates).")
print("نحتاجو نعدلوها بس لو ضفنا طرق مختصرة جديدة أو نقصنا من تكلفة الحركات.\n")

print("--- تنفيذ مهمة A* ---")
start = initial_state()
target = 'D'
astar_route = a_star(start, target)

if astar_route is not None:
    print(f"تم إيجاد مسار في {len(astar_route)} خطوات:")
    for step in astar_route:
        print(f" -> {step}")
else:
    print("لم يتم إيجاد مسار.")

تفسير A*:
الـ g(n) يمثل التكلفة الفعلية من نقطة البداية للحالة الحالية n.
الـ h(n) يمثل التكلفة التقديرية (الـ heuristic) من الحالة n للهدف.
قيمة h المعطاة تعتبر مقبولة (admissible) لأنها تمثل أقصر مسار في الخريطة الأصلية. وبما أن مسارنا انحذف منه طريق (E-G)، المسافات الفعلية يا إما تقعد زي ما هي أو تزيد، وبالتالي قيمة h مستحيل تبالغ في التقدير (never overestimates).
نحتاجو نعدلوها بس لو ضفنا طرق مختصرة جديدة أو نقصنا من تكلفة الحركات.

--- تنفيذ مهمة A* ---
تم إيجاد مسار في 3 خطوات:
 -> ('Move', 'A')
 -> ('Move', 'D')
 -> ('Clean', 'D')


## 10. Sources, assistance and questions

Cite adapted course code and explain your changes. Disclose AI tool/help/verification or state none used. Record any unresolved questions. Prepare to explain BFS, A*, the heuristic and your simulator after the professor returns.


Sources and Adapted Code:

اعتمدت على الهيكل العام لخوارزميات البحث (BFS و A*) من أمثلة المنهج (Course Code). التعديل الأساسي اللي درته هو تكييف الخوارزمية باش تتعامل مع "حالة مركبة" (Tuple) تتكون من (المكان، حالة الغرفة) بدل من مجرد عقدة (Node) عادية، بالإضافة لإضافة حركة "التنظيف" (Clean) كجزء من المسار والتأكد من تحديث الحالة.

AI Tool Disclosure:

نعم، استخدمت أداة ذكاء اصطناعي كـ (مساعد) لمراجعة منطق المحاكي (Simulator)، والمساعدة في تصحيح بعض الأخطاء البرمجية (Debugging) في خوارزمية A*، بالإضافة لتنسيق الشرح النظري.

Unresolved Questions:

لا توجد أسئلة أو مشاكل معلقة. المحاكي والخوارزميات يعملان بنجاح ويستخرجان المسار الأمثل. مستعد لشرح الخوارزميات (BFS, A*) والـ Heuristics والمحاكي لأستاذ متى طُلب ذلك.

## GitHub + Colab workflow

1. On the public starter, use Use this template > Create a new repository, choose Private, then invite the professor using the announced GitHub username.
2. Your copy already includes project.ipynb. Open it in Colab using authorized GitHub access or download/upload.
3. Save each work session explicitly to GitHub: File > Save a copy in/to GitHub, correct repo/branch and project.ipynb. Drive autosave alone does not update GitHub.
4. If integration fails, download the notebook from Colab, upload it to GitHub to replace project.ipynb, and commit. No terminal or embedded tokens are needed.
5. Commit real progress throughout the project. Complete README.md using the supplied template.
6. Reopen the committed notebook in a fresh Colab session and verify it runs. Submit repo URL + final commit URL in Classroom and Turn in.

See the student guide pages 5-6 and docs/github-colab-guide.md for details. The professor must accept the invitation before viewing private work; do not wait for acceptance to begin.


## 11. Submission checklist

- [x] Identity, chosen project and generated variant recorded
- [x] Task 1: PEAS and environment explained
- [x] Task 2: state, legal actions, transition, goal and costs defined
- [x] Task 4: legal movement, service and task goal implemented
- [x] Task 5: working BFS and assigned mission output
- [x] Task 6: working A*, admissible heuristic explanation and assigned mission output
- [x] Sources and assistance disclosed
- [x] README with task/variant/files/dependencies/run steps
- [x] Clean restart/run-all checked and saved
- [x] project.ipynb and README.md saved in my private GitHub repository
- [x] Professor invited; repository URL and final commit URL submitted in Classroom

Original task numbers are retained for reference. Tasks 3, 7, 8 and 9 are removed. There is no Day 7 checkpoint.